# Evidencia de Aprendizaje 3 — Optimización de procesos de desarrollo

**Programa:** Ingeniería de Software y Datos
**Materia:** Programación para Análisis de Datos — PREICA2602B020048
**Docente:** Andrés Felipe Palacio
**Estudiante:** Jesus Gonzalez
**Fecha:** noviembre de 2026

---

## 1. Introducción

Este cuaderno acompaña el informe de la tercera evidencia y ejecuta sus partes
verificables. El proyecto continuó la cadena de las dos evidencias anteriores:

| Evidencia | Aporte | Técnica central |
|---|---|---|
| EA1 | Catálogo de 16 teléfonos de Amazon.com | *Web scraping* con BeautifulSoup, Selenium y Scrapy |
| EA2 | Panel estático construido y publicado con CI/CD | Git, GitHub Actions y GitHub Pages |
| EA3 | Servicio web dinámico con API JSON | Flask, GitHub Actions y Render |

El tema de interés es la **automatización del ciclo de desarrollo**: un mismo
catálogo de datos que en la EA1 solo existía como archivo CSV, en la EA2 se
volvió una página publicada y aquí se convierte en un **servicio consultable**,
mantenido por un flujo que prueba cada cambio y publica la rama principal sin
intervención manual.

## 2. Objetivos

**Objetivo general.** Automatizar el desarrollo y despliegue de una aplicación
web que sirve el catálogo de la EA1, usando GitHub Actions para la integración
continua y Render como servicio de alojamiento en la nube.

**Objetivos específicos.**

1. Convertir el catálogo en un servicio Flask con página web y API JSON.
2. Mantener el esquema de ramas con *pull requests* verificados por CI.
3. Ampliar la matriz de integración continua a dos sistemas y dos versiones de Python.
4. Configurar el despliegue continuo hacia Render y verificar la versión publicada.
5. Documentar el flujo DevOps en el repositorio y en el informe del trabajo.

## 3. Metodología de *scraping* (resumen de la EA1)

El catálogo que alimenta este servicio proviene de la primera evidencia. La
extracción combinó tres métodos sobre la misma búsqueda de Amazon.com
(contexto de entrega ZIP 10001, Nueva York):

- **Selenium** fijó la ubicación de entrega con el widget *Deliver to* y
  exportó las cookies de sesión, resolviendo el bloqueo del *buybox* que
  Amazon aplica a visitantes fuera de Estados Unidos.
- **BeautifulSoup** reutilizó esas cookies para un parseo rápido y masivo del
  HTML de las fichas.
- **Scrapy** recorrió la búsqueda con concurrencia controlada y pipelines.

Los tres métodos observaron el mismo estado comercial; la comparación dio una
concordancia del 100 % en los campos clave y una cobertura del 98,8 %. El
resultado consolidado es el CSV que este servicio carga al arrancar:

**Artículo principal:** Samsung Galaxy A57 5G (128 GB), ASIN `B0GW9TVHJB`,
USD 499,99, calificación 4,4 con 209 reseñas.

## 4. El catálogo y el servicio

Carga del catálogo con el mismo módulo de validación que usa la aplicación:
si el CSV no cumple el contrato (ASIN, título, marca y campos numéricos
consistentes), el servicio no arranca.

In [1]:
from src.servicio.datos import cargar

productos = cargar("data/catalogo_consolidado.csv")
print(f"Productos cargados: {len(productos)}")
principal = next(p for p in productos if p.asin == "B0GW9TVHJB")
print(f"Artículo principal: {principal.marca} · ${principal.precio} · "
      f"calificación {principal.calificacion} · {principal.resenas} reseñas")

Productos cargados: 16
Artículo principal: Samsung · $499.99 · calificación 4.4 · 209 reseñas


### 4.1 La aplicación y sus endpoints

La fábrica `create_app` construye la aplicación Flask con el catálogo ya
validado. Aquí se prueba con el cliente de pruebas de Flask, que ejecuta las
mismas rutas que el servicio publicado, sin abrir un puerto.

In [2]:
from src.servicio import create_app

cliente = create_app("data/catalogo_consolidado.csv").test_client()

respuesta = cliente.get("/healthz")
print("Estado:", respuesta.status_code, respuesta.get_json())

Estado: 200 {'productos': 16, 'revision': 'local', 'status': 'ok'}


In [3]:
# La página servida en la raíz: título, cifras y primer renglón de la tabla
html = cliente.get("/").data.decode("utf-8")
for fragmento in ("Catálogo de teléfonos inteligentes", "B0GW9TVHJB",
                  "Samsung Galaxy A57 5G", "API JSON"):
    print(f"{fragmento!r:>45}: {'presente' if fragmento in html else 'FALTA'}")

         'Catálogo de teléfonos inteligentes': presente
                                 'B0GW9TVHJB': presente
                      'Samsung Galaxy A57 5G': presente
                                   'API JSON': presente


In [4]:
import json

# Filtro por marca y orden por precio descendente
r = cliente.get("/api/productos?marca=Samsung&orden=precio_desc").get_json()
print(f"Samsung: {r['total']} productos")
for p in r["productos"][:4]:
    print(f"  ${p['precio']:>7.2f}  {p['titulo'][:58]}")

Samsung: 10 productos
  $ 599.99  Samsung Galaxy S26 FE, Unlocked Phone, Graphite, 128GB
  $ 499.99  Samsung Galaxy A57 5G (2026), Unlocked Android Smartphone,
  $ 489.99  Samsung Galaxy A37 5G (2026), Unlocked Android Smartphone,
  $ 244.99  Samsung Galaxy A17 5G Smart Phone 128GB US 1 Yr Manufactur


In [5]:
# Búsqueda de texto y rango de precios
r = cliente.get("/api/productos?q=galaxy&min_precio=400&max_precio=700").get_json()
print(f"Galaxy entre $400 y $700: {r['total']} productos")
for p in r["productos"]:
    print(f"  ${p['precio']:>7.2f}  {p['titulo'][:58]}")

Galaxy entre $400 y $700: 3 productos
  $ 489.99  Samsung Galaxy A37 5G (2026), Unlocked Android Smartphone,
  $ 499.99  Samsung Galaxy A57 5G (2026), Unlocked Android Smartphone,
  $ 599.99  Samsung Galaxy S26 FE, Unlocked Phone, Graphite, 128GB


In [6]:
# Manejo de errores: parámetro inválido (400) y ASIN inexistente (404)
r400 = cliente.get("/api/productos?min_precio=abc")
r404 = cliente.get("/api/productos/XXXXXXXXXXXX")
print(f"min_precio=abc -> {r400.status_code}: {r400.get_json()['error']}")
print(f"ASIN falso     -> {r404.status_code}: {r404.get_json()['error']}")

min_precio=abc -> 400: El parámetro 'min_precio' espera un número y recibió 'abc'
ASIN falso     -> 404: No existe un producto con el ASIN 'XXXXXXXXXXXX'


In [7]:
# Métricas del mercado expuestas por la API
m = cliente.get("/api/metricas").get_json()
print("Resumen:", json.dumps(m["resumen"], indent=2, ensure_ascii=False))
print("\nDistribución por tramo de precio:")
for t in m["distribucion_precio"]:
    print(f"  {t['tramo']:>12}: {t['productos']}")

Resumen: {
  "calificacion_promedio": 4.19,
  "con_precio": 15,
  "descuento_promedio": 12.4,
  "marcas": 7,
  "precio_maximo": 599.99,
  "precio_mediano": 166.29,
  "precio_minimo": 69.99,
  "precio_promedio": 233.95,
  "productos": 16,
  "resenas_totales": 24865
}

Distribución por tramo de precio:
        < $300: 12
   $300 - $499: 2
   $500 - $799: 1
       >= $800: 0


## 5. Automatización con GitHub Actions

El repositorio define dos flujos. El de **integración continua** instala las
dependencias, compila los módulos y ejecuta las 41 pruebas sobre una matriz de
Ubuntu y Windows con Python 3.12 y 3.13. El de **despliegue continuo** completa
la publicación de Render verificando que `/healthz` reporte el SHA del commit
fusionado. Aquí se leen los archivos tal como los ve Actions:

In [8]:
import yaml

for nombre in ("ci.yml", "cd.yml"):
    ruta = f".github/workflows/{nombre}"
    flujo = yaml.safe_load(open(ruta, encoding="utf-8"))
    # Detalle conocido de PyYAML: la clave 'on' se parsea como el booleano True
    # (YAML 1.1). Por eso la búsqueda contempla ambas formas.
    activadores = flujo.get("on") or flujo.get(True) or {}
    trabajos = flujo["jobs"]
    pasos = sum(len(t["steps"]) for t in trabajos.values())
    print(f"{nombre}: {len(trabajos)} trabajo(s), {pasos} pasos, se activa en: {list(activadores)}")
    for id_trabajo, trabajo in trabajos.items():
        print(f"  [{id_trabajo}] {trabajo.get('name', id_trabajo)}")

ci.yml: 1 trabajo(s), 5 pasos, se activa en: ['push', 'pull_request']


  [pruebas] Pruebas (${{ matrix.os }}, Python ${{ matrix.python }})


cd.yml: 1 trabajo(s), 3 pasos, se activa en: ['push', 'workflow_dispatch']
  [desplegar] Publicar en Render y verificar


In [9]:
# El blueprint de Render: lo que el servicio ejecuta en la nube
blueprint = yaml.safe_load(open("render.yaml", encoding="utf-8"))
servicio = blueprint["services"][0]
for clave in ("name", "runtime", "plan", "buildCommand", "startCommand",
              "healthCheckPath", "autoDeploy"):
    print(f"{clave:>16}: {servicio.get(clave)}")

            name: ea3-catalogo-api


         runtime: python
            plan: free
    buildCommand: pip install -r requirements.txt
    startCommand: gunicorn wsgi:app --workers 2 --threads 4 --bind 0.0.0.0:$PORT
 healthCheckPath: /healthz
      autoDeploy: True


### 5.1 Pruebas automatizadas

La suite completa corre en cada *pull request*; aquí se ejecuta en el cuaderno
para evidenciar que el estado entregado está verde:

In [10]:
import unittest

cargador = unittest.TestLoader()
suite = cargador.discover("tests")
resultado = unittest.TextTestRunner(verbosity=0).run(suite)
print(f"Pruebas ejecutadas: {resultado.testsRun}")
print(f"Errores: {len(resultado.errors)} · Fallos: {len(resultado.failures)}")
print("Veredicto:", "APROBADO" if resultado.wasSuccessful() else "RECHAZADO")

----------------------------------------------------------------------
Ran 41 tests in 0.875s

OK


Pruebas ejecutadas: 41
Errores: 0 · Fallos: 0
Veredicto: APROBADO


## 6. Verificación del despliegue en Render

El servicio queda publicado en `https://ea3-catalogo-api.onrender.com`.
Las dos celdas siguientes consultan el servicio **real**; si el plan gratuito
lo suspendió por inactividad, la primera petición tarda unos segundos en
despertarlo.

In [11]:
import requests

URL = "https://ea3-catalogo-api.onrender.com"

salud = requests.get(f"{URL}/healthz", timeout=90).json()
print("Estado del servicio:", salud["status"])
print("Commit desplegado:", salud["revision"])
print("Productos servidos:", salud["productos"])

Estado del servicio: ok
Commit desplegado: 700db5b
Productos servidos: 16


In [12]:
# Consulta real a la API publicada: top Samsung por precio descendente
publico = requests.get(f"{URL}/api/productos",
                       params={"marca": "Samsung", "orden": "precio_desc"},
                       timeout=90).json()
print(f"Samsung en producción: {publico['total']} productos")
for p in publico["productos"][:3]:
    print(f"  ${p['precio']:>7.2f}  {p['titulo'][:58]}")

metricas = requests.get(f"{URL}/api/metricas", timeout=90).json()
print("\nPrecio promedio en producción:", metricas["resumen"]["precio_promedio"])

Samsung en producción: 10 productos
  $ 599.99  Samsung Galaxy S26 FE, Unlocked Phone, Graphite, 128GB
  $ 499.99  Samsung Galaxy A57 5G (2026), Unlocked Android Smartphone,
  $ 489.99  Samsung Galaxy A37 5G (2026), Unlocked Android Smartphone,



Precio promedio en producción: 233.95


## 7. Resultados y conclusiones

**Resultados.**

1. El catálogo de la EA1 se transformó en un servicio consultable: página HTML
   con las cifras del mercado y una API JSON con filtros por marca, búsqueda,
   rango de precios y cinco órdenes, con errores de consulta manejados (400 y
   404 con cuerpo JSON).
2. La automatización quedó completa: 41 pruebas sobre una matriz de cuatro
   combinaciones (Ubuntu/Windows × Python 3.12/3.13), *pull requests*
   verificados antes de fusionar y despliegue continuo a Render con
   verificación de la versión publicada vía `/healthz` y el SHA del commit.
3. La cadena EA1 → EA2 → EA3 muestra la evolución de un mismo conjunto de
   datos por tres niveles de madurez: archivo → página estática → servicio web
   automatizado.

**Conclusiones.**

- La automatización cambia la forma de trabajar: el costo de publicar un
  cambio baja a "fusionar el *pull request*", y la verificación automática de
  la versión publicada elimina la duda clásica de "¿qué está corriendo en
  producción?".
- La matriz de pruebas detectó temprano las dependencias del entorno: el
  código que corre igual en Windows y Ubuntu, con dos versiones de Python, es
  código portable de verdad.
- El *web scraping* del trabajo original sigue siendo la base: sin un catálogo
  limpio y validado, ningún pipeline posterior tendría datos confiables que
  servir.